# 4 — AI Lead Qualifier
> Score any inbound sales lead against your Ideal Customer Profile and get a tier (hot / warm / cold), a score out of 10, and the exact criteria the lead met or missed — in one call.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI


class LeadScore(BaseModel):
    company: str
    score: int = Field(ge=1, le=10, description="ICP fit score from 1 (no fit) to 10 (perfect fit)")
    tier: Literal["hot", "warm", "cold"]
    criteria_met: List[str]
    criteria_missed: List[str]
    recommended_action: str
    reasoning: str


ICP_RUBRIC = SystemMessage("""
You are a sales qualification assistant. Score inbound leads against this ICP rubric.

IDEAL CUSTOMER PROFILE (ICP):
  Industry:      SaaS, FinTech, or E-commerce
  Company size:  50-500 employees
  Pain point:    manual workflows, data silos, or compliance burden
  Buyer role:    VP Operations, CFO, or CTO
  Budget signal: existing software spend > $5k/month

SCORING:
  8-10 -> hot   (3+ criteria met, strong pain + budget signal)
  5-7  -> warm  (2 criteria met, or strong pain but budget unclear)
  1-4  -> cold  (fewer than 2 criteria met)

Rules:
- Populate criteria_met and criteria_missed by naming the exact ICP criteria above.
- reasoning must explain the score in one or two sentences citing the criteria.
- Never invent data not present in the lead description.
""")


def run(lead_description: str) -> LeadScore:
    llm = ChatOpenAI(model="gpt-4o-mini")
    workflow = ICP_RUBRIC | llm.with_structured_output(LeadScore)
    return workflow.invoke(HumanMessage(lead_description))

## The scenario

We received an inbound inquiry from a 200-person e-commerce logistics company. Their CTO flagged growing compliance headaches as they expand into the EU — but they were vague about current software spend. The agent will score this lead against our ICP, flag what is missing, and tell us exactly what to do next.

In [ ]:
LEAD = """
Company:  NorthBridge Logistics
Industry: E-commerce logistics
Size:     200 employees
Contact:  David Park, CTO

Notes:
David reached out after attending our webinar. Their team is grappling with
EU GDPR compliance across four data warehouses that don't talk to each other --
each region maintains its own exports and reconciliation process manually.
He mentioned they're evaluating several vendors and hinted at budget approval
for Q4. No concrete spend figure mentioned, but they have a dedicated data team
of 8 engineers which implies meaningful tooling investment.
"""

result = run(LEAD)

tier_label = {"hot": "HOT", "warm": "WARM", "cold": "COLD"}[result.tier]

print(f"{'='*52}")
print(f"  {result.company}")
print(f"  Score: {result.score}/10   Tier: {tier_label}")
print(f"{'='*52}")
print(f"\nACTION:  {result.recommended_action}")
print("\nCRITERIA MET")
for c in result.criteria_met:
    print(f"  +  {c}")
print("\nCRITERIA MISSED")
for c in result.criteria_missed:
    print(f"  -  {c}")
print("\nWHY THIS SCORE")
print(f"  {result.reasoning}")

## Use your own data

Replace the `LEAD` string above with:
- Your CRM notes or a copied email thread from the prospect
- Any combination of company name, size, industry, buyer role, and pain points you have on hand

The agent will return a score, a tier, and the specific ICP criteria your lead satisfies — so you know exactly what to probe on your next call.